In [1]:
import psutil
from tqdm import tqdm

def ram_usage():
    vm = psutil.virtual_memory()
    return f"RAM: {vm.used/(1024**3):.1f}/{vm.total/(1024**3):.1f} GB ({vm.percent:.0f}%)"
print(f"{ram_usage()}")

RAM: 4.3/7.8 GB (55%)


## 0 · Imports & config

In [ ]:
import os, gc, sys, logging, json, re, shutil, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
from tqdm.auto import tqdm

logging.basicConfig(format='%(asctime)s  %(levelname)-8s  %(message)s',
                    level=logging.INFO, datefmt='%H:%M:%S')
log = logging.getLogger('preprocess')


ROOT_DIR      = Path(r"D:\\Documents\\Project\\Data Mining")
REVIEW_PATH   = Path(r"D:\Documents\Project\Data Mining\Home_and_Kitchen.jsonl\Home_and_Kitchen.jsonl")
META_PATH     = Path(r"D:\Documents\Project\Data Mining\meta_Home_and_Kitchen.jsonl\meta_Home_and_Kitchen.jsonl")

PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
CLEANED_DIR   = ROOT_DIR / 'data' / 'cleaned'         
VOCAB_DIR     = ROOT_DIR / 'data' / 'vocab'             
FIGURES_DIR   = ROOT_DIR / 'outputs' / 'figures'
TEMP_DIR      = PROCESSED_DIR / '_temp_chunks'

META_FILE     = PROCESSED_DIR / 'meta_clean.parquet'  

for d in [PROCESSED_DIR, CLEANED_DIR, VOCAB_DIR, FIGURES_DIR, TEMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)


CHUNK_SIZE            = 500_000
MIN_USER_INTERACTIONS = 5       
MIN_ITEM_INTERACTIONS = 5       
MAX_SEQ_LEN           = 200    
RANDOM_SEED           = 42
np.random.seed(RANDOM_SEED)

print('=== CẤU HÌNH & THÔNG TIN FILE ===')
print(f'   Chunk size            : {CHUNK_SIZE:,} dòng/lần')
print(f'   Min user interactions : {MIN_USER_INTERACTIONS}')
print(f'   Min item interactions : {MIN_ITEM_INTERACTIONS}')
print(f'   Max seq len           : {MAX_SEQ_LEN}')
print(f'   Processed dir         : {PROCESSED_DIR}')
print(f'   Cleaned  dir          : {CLEANED_DIR}')

for label, path in [('Review JSONL', REVIEW_PATH), ('Meta JSONL', META_PATH)]:
    if path.exists():
        print(f'   {label}: {path.stat().st_size/(1024**3):.2f} GB')
    else:
        print(f'   {label}: KHÔNG TÌM THẤY tại {path}')

print(f'   {ram_usage()}')

## 1 · Load reviews (lazy)

In [ ]:
gc.collect()

In [ ]:
import json
import gc
import psutil
import shutil
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from tqdm.auto import tqdm


import logging
logging.basicConfig(format="%(asctime)s  %(levelname)-8s  %(message)s",
                    level=logging.INFO, datefmt="%H:%M:%S")
log = logging.getLogger("preprocess")

log.info("Bước 1/3: Đọc JSONL và xả liên tục ra ổ cứng để ép RAM...")

TEMP_DIR = PROCESSED_DIR / "_temp_chunks"
TEMP_DIR.mkdir(parents=True, exist_ok=True)


for f in TEMP_DIR.glob("chunk_*.parquet"):
    f.unlink()

CHUNK_SIZE = 500_000
current_chunk = []
chunk_id = 0
total_raw = 0


with open(REVIEW_PATH, 'rt', encoding='utf-8') as f:
    pbar = tqdm(f, desc="Processing JSONL", unit=" lines")
    
    for line in pbar:
        total_raw += 1
        try:
            data = json.loads(line.strip())
        except json.JSONDecodeError:
            continue
            
       
        if not data.get("verified_purchase", False):
            continue
            
        u = data.get("user_id")
        it = data.get("parent_asin")
        r = data.get("rating")
        ts = data.get("timestamp")
        
        if not all([u, it, r, ts]):
            continue

        current_chunk.append({
            "user_id": u,
            "parent_asin": it,
            "rating": r,
            "timestamp": ts
        })
        
 
        if len(current_chunk) >= CHUNK_SIZE:
            df_chunk = pd.DataFrame(current_chunk)
            
        
            df_chunk = df_chunk.sort_values("timestamp", ascending=False)
            df_chunk = df_chunk.drop_duplicates(subset=["user_id", "parent_asin"], keep="first")
            
            chunk_path = TEMP_DIR / f"chunk_{chunk_id:04d}.parquet"
            df_chunk.to_parquet(chunk_path, index=False)
            
            chunk_id += 1
            current_chunk = [] 
            del df_chunk
            gc.collect()
            
            pbar.set_postfix({'Chunks': chunk_id, 'RAM': f"{psutil.virtual_memory().percent}%"})


if current_chunk:
    df_chunk = pd.DataFrame(current_chunk)
    df_chunk = df_chunk.sort_values("timestamp", ascending=False)
    df_chunk = df_chunk.drop_duplicates(subset=["user_id", "parent_asin"], keep="first")
    df_chunk.to_parquet(TEMP_DIR / f"chunk_{chunk_id:04d}.parquet", index=False)
    del df_chunk, current_chunk
    gc.collect()

log.info("Bước 2/3: Gộp luồng các chunk bằng ParquetWriter ...")

chunk_files = sorted(TEMP_DIR.glob("chunk_*.parquet"))
output_path = PROCESSED_DIR / "review_clean_temp.parquet"

writer = None
for chunk_file in tqdm(chunk_files, desc="Merging chunks"):
    table = pq.read_table(chunk_file)
    
    if writer is None:
        writer = pq.ParquetWriter(output_path, table.schema, compression='snappy')
        
    writer.write_table(table)
    del table
    gc.collect()

if writer:
    writer.close()


shutil.rmtree(TEMP_DIR)

log.info("Bước 3/3: Bắt đầu load file đã gộp vào RAM...")

df = pd.read_parquet(output_path, engine="pyarrow")

df['user_id']     = df['user_id'].astype(str)
df['parent_asin'] = df['parent_asin'].astype(str)
df['rating']      = pd.to_numeric(df['rating'], errors='coerce').astype('float32')
df['timestamp']   = pd.to_numeric(df['timestamp'], errors='coerce').astype('int64')
log.info(f"Đã load {len(df):,} dòng. Đang sắp xếp (Sort) timestamp in-place...")
df.sort_values("timestamp", ascending=False, inplace=True)
gc.collect()
log.info("Sort xong! Đang xóa trùng lặp (Drop Duplicates)...")

df.drop_duplicates(subset=["user_id", "parent_asin"], keep="first", inplace=True)
gc.collect()

log.info("Xóa trùng lặp xong! Đang format ngày tháng...")
df.reset_index(drop=True, inplace=True)

ts_unit = 'ms' if df['timestamp'].max() > 1_000_000_000_000 else 's'
df['dt'] = pd.to_datetime(df['timestamp'], unit=ts_unit)
log.info(f'Timestamp unit detected: {ts_unit}')
if output_path.exists():
    output_path.unlink()

print("\n=== KẾT QUẢ XỬ LÝ HOÀN TẤT ===")
print(f"  Tổng dòng đọc gốc : {total_raw:,}")
print(f"  Dòng giữ lại      : {len(df):,}")
print(f"  Date range        : {df['dt'].min().date()} → {df['dt'].max().date()}")


## 2 · Clean Reviews
### 2.1 Load + dedup + filter verified

### 2.2 K-core filtering
Lặp lọc đến khi hội tụ: user ≥ 5 tương tác **và** item ≥ 5 tương tác.

In [ ]:
import gc
from tqdm.auto import tqdm

def kcore_filter(df: pd.DataFrame, min_u: int, min_i: int) -> pd.DataFrame:
    """
    Lặp lọc user/item có ít tương tác đến khi kích thước không đổi.
    Thường hội tụ sau 3-5 vòng.
    """
    prev_len = -1
    iteration = 0
    
   
    pbar = tqdm(desc="K-core Filtering", unit=" iter")
    
    while len(df) != prev_len:
        prev_len = len(df)
        iteration += 1
        
        item_counts = df["parent_asin"].value_counts()
        df = df[df["parent_asin"].isin(item_counts[item_counts >= min_i].index)]
        
     
        user_counts = df["user_id"].value_counts()
        df = df[df["user_id"].isin(user_counts[user_counts >= min_u].index)]
        
        
        pbar.update(1)
        pbar.set_postfix({
            'Rows':  f'{len(df):,}',
            'Users': f'{len(user_counts):,}',
            'Items': f'{len(item_counts):,}',
            'RAM':   f'{psutil.virtual_memory().percent:.0f}%',
        })
      
        del item_counts, user_counts
        gc.collect()
        
    pbar.close()
    print(f"  Converged after {iteration} iterations.")
    return df.reset_index(drop=True)


print(f"Before k-core: {len(df):,} rows | "
      f"{df['user_id'].value_counts().size:,} users | "
      f"{df['parent_asin'].value_counts().size:,} items)")

df = kcore_filter(df, MIN_USER_INTERACTIONS, MIN_ITEM_INTERACTIONS)

print(f"\nFinal      : {len(df):,} rows | "
      f"{df['user_id'].value_counts().size:,} users | "
      f"{df['parent_asin'].value_counts().size:,} items)")
print(psutil.virtual_memory())

## 3 · Clean Meta
Parse price, dedup `parent_asin`, filter chỉ giữ item có trong reviews.

In [ ]:
import re

log.info("Loading meta...")
META_COLS = ['parent_asin', 'title', 'price', 'description', 'cat_level_1', 'cat_level_2', 'cat_level_3', 'average_rating', 'main_category', 'images']


_meta_folder = PROCESSED_DIR / 'meta'
if _meta_folder.is_dir() and any(_meta_folder.glob('*.parquet')):
    df_meta = pd.read_parquet(_meta_folder, columns=META_COLS, engine='pyarrow')
else:
    df_meta = pd.read_parquet(META_FILE, columns=META_COLS, engine='pyarrow')

df_meta['parent_asin']    = df_meta['parent_asin'].astype(str)
df_meta['average_rating'] = pd.to_numeric(df_meta['average_rating'], errors='coerce').astype('float32')
df_meta['rating_number']  = pd.to_numeric(df_meta['rating_number'], errors='coerce').fillna(0).astype('int32')
print(f"Loaded meta: {len(df_meta):,} rows  |  {ram_usage()}")


df_meta = df_meta.drop_duplicates(subset=["parent_asin"], keep="first").reset_index(drop=True)
print(f"After dedup: {len(df_meta):,} items")


def parse_price(s) -> float:
    """'$12.99' → 12.99  |  'None'/'' → NaN  |  '10.99 - 20.99' → mean"""
    if pd.isna(s) or str(s).strip() in ("", "None", "null"):
        return float("nan")
    nums = re.findall(r"[\d]+(?:\.[\d]+)?", str(s))
    if not nums:
        return float("nan")
    vals = [float(n) for n in nums]
    return round(sum(vals) / len(vals), 2)  

df_meta["price_usd"] = df_meta["price"].apply(parse_price)
price_null = df_meta["price_usd"].isna().sum()
print(f"Price parsed: {len(df_meta)-price_null:,} valid | {price_null:,} NaN")
print(f"Price range : ${df_meta['price_usd'].min():.2f} – ${df_meta['price_usd'].quantile(0.99):.2f} (p99)")


valid_items = set(df["parent_asin"].unique())
df_meta = df_meta[df_meta["parent_asin"].isin(valid_items)].reset_index(drop=True)
print(f"After inner join: {len(df_meta):,} meta items  ({len(valid_items):,} unique in reviews)")
print(ram_usage())

## 4 · Build Item Vocab
Map `parent_asin` (string) → `item_idx` (int) — cần thiết cho Item2Vec và GRU4Rec.

In [ ]:

item_freq  = df["parent_asin"].value_counts()   
item2idx   = {item: idx for idx, item in enumerate(item_freq.index)}
idx2item   = {idx: item for item, idx in item2idx.items()}
N_ITEMS    = len(item2idx)


user_freq  = df["user_id"].value_counts()
user2idx   = {u: i for i, u in enumerate(user_freq.index)}
N_USERS    = len(user2idx)


df["item_idx"] = df["parent_asin"].map(item2idx)
df["user_idx"] = df["user_id"].map(user2idx)


df_meta["item_idx"] = df_meta["parent_asin"].map(item2idx)

print(f"Vocab size  : {N_ITEMS:,} items  |  {N_USERS:,} users")


import json
vocab_dir = ROOT_DIR / "data" / "vocab"
vocab_dir.mkdir(exist_ok=True)
with open(vocab_dir / "item2idx.json", "w") as f:
    json.dump(item2idx, f)
with open(vocab_dir / "idx2item.json", "w") as f:
    json.dump({str(k): v for k, v in idx2item.items()}, f)
print(f"Vocab saved → {vocab_dir}")

## 5 · Build User Sessions
Group by `user_id`, sort theo `timestamp` tăng dần → ra sequence `[item_idx_0, item_idx_1, ...]`.

In [ ]:
print('Building sessions...')


df_sorted = df.sort_values(["user_idx", "timestamp"], ascending=True)


sessions = (
    df_sorted.groupby("user_idx", sort=False)["item_idx"]
    .apply(list)
    .reset_index()
    .rename(columns={"item_idx": "item_seq"})
)

sessions["item_seq"] = sessions["item_seq"].apply(
    lambda seq: seq[-MAX_SEQ_LEN:] if len(seq) > MAX_SEQ_LEN else seq
)
sessions["seq_len"] = sessions["item_seq"].apply(len)

print(f"Total sessions : {len(sessions):,}")
print(f"Sequence length:")
print(sessions["seq_len"].describe().to_string())
print(ram_usage())

## 6 · Temporal Train / Val / Test Split
**Leave-One-Out (LOO) temporal:**  
- `test`  = item **cuối cùng** trong sequence mỗi user  
- `val`   = item **áp cuối**  
- `train` = tất cả phần còn lại  

Đây là chuẩn phổ biến nhất trong session-based recommendation.

In [ ]:

sessions = sessions[sessions["seq_len"] >= 3].reset_index(drop=True)
print(f"Sessions with len>=3: {len(sessions):,}")


sessions["train_seq"] = sessions["item_seq"].apply(lambda s: s[:-2])
sessions["val_item"]  = sessions["item_seq"].apply(lambda s: s[-2])
sessions["test_item"] = sessions["item_seq"].apply(lambda s: s[-1])


total_interactions = sessions["seq_len"].sum()
train_interactions = sessions["train_seq"].apply(len).sum()
print(f"\nSplit summary:")
print(f"  train interactions : {train_interactions:,}  ({train_interactions/total_interactions*100:.1f}%)")
print(f"  val  interactions  : {len(sessions):,}  ({len(sessions)/total_interactions*100:.1f}%)")
print(f"  test interactions  : {len(sessions):,}  ({len(sessions)/total_interactions*100:.1f}%)")
print(f"  total              : {total_interactions:,}")

## 7 · Save outputs

In [ ]:

clean_cols = ["user_idx", "user_id", "item_idx", "parent_asin",
              "rating", "timestamp", "dt"]
df[clean_cols].to_parquet(CLEANED_DIR / "reviews_clean.parquet",
                           index=False, engine="pyarrow")
print(f"Saved reviews_clean.parquet  ({len(df):,} rows)")


df_meta.to_parquet(CLEANED_DIR / "meta_clean.parquet",
                   index=False, engine="pyarrow")
print(f"Saved meta_clean.parquet     ({len(df_meta):,} rows)")

import pickle
sessions_save = sessions[["user_idx", "train_seq", "val_item",
                           "test_item", "seq_len"]].copy()


with open(CLEANED_DIR / "sessions.pkl", "wb") as f:
    pickle.dump(sessions_save, f)
print(f"Saved sessions.pkl           ({len(sessions_save):,} users)")


records = []
for row in tqdm(sessions_save.itertuples(), total=len(sessions_save), desc="Flatten"):
    for pos, item in enumerate(row.train_seq):
        records.append((row.user_idx, pos, item, "train"))
    records.append((row.user_idx, len(row.train_seq), row.val_item, "val"))
    records.append((row.user_idx, len(row.train_seq)+1, row.test_item, "test"))

df_flat = pd.DataFrame(records, columns=["user_idx", "position", "item_idx", "split"])
df_flat.to_parquet(CLEANED_DIR / "interactions_flat.parquet",
                   index=False, engine="pyarrow")
print(f"Saved interactions_flat.parquet  ({len(df_flat):,} rows)")
del records, df_flat; gc.collect()

──
print("\n" + "="*50)
print("  PREPROCESSING COMPLETE")
print("="*50)
print(f"  Users          : {N_USERS:>10,}")
print(f"  Items          : {N_ITEMS:>10,}")
print(f"  Interactions   : {len(df):>10,}")
print(f"  Sessions       : {len(sessions_save):>10,}")
print(f"  Avg seq len    : {sessions_save['seq_len'].mean():>10.2f}")
print(f"  Sparsity       : {1 - len(df)/(N_USERS*N_ITEMS):>10.6f}")
print("="*50)
print(ram_usage())